# Attention Flow Analysis

Advanced attention flow analysis: information routing paths, bottleneck detection, cross-layer coherence, and source attribution.

**References:**
- Abnar & Zuidema (2020) "Quantifying Attention Flow in Transformers"
- Brunner et al. (2020) "On Identifiability in Transformers"

## Why This Matters

Attention flow analysis traces information paths through the network by composing attention patterns across layers. Flow paths, bottleneck detection, and source attribution reveal the multi-layer routing of information that single-layer attention patterns cannot capture.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_flow_analysis import (
    attention_flow_paths,
    attention_bottleneck_detection,
    cross_layer_attention_coherence,
    attention_source_attribution,
    attention_information_routing,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads, seq_len={len(tokens)}")

In [ ]:
# 1. Attention flow paths
flow = attention_flow_paths(model, tokens, source_pos=0, target_pos=-1)
print(f"Total flow from pos 0 to final: {flow['total_flow']:.4f}")
print(f"Dominant layer: {flow['dominant_layer']}")
print(f"\nPer-head flow (source=0 -> target=-1):")
for l in range(cfg.n_layers):
    scores = [f"{flow['per_head_flow'][l,h]:.4f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {', '.join(scores)}")

In [ ]:
# 2. Attention bottleneck detection
bn = attention_bottleneck_detection(model, tokens)
print(f"Global bottleneck position: {bn['global_bottleneck']}")
for l in range(cfg.n_layers):
    has = 'YES' if bn['has_bottleneck'][l] else 'no'
    print(f"  Layer {l}: pos={bn['bottleneck_positions'][l]}, strength={bn['bottleneck_strength'][l]:.4f} ({has})")

In [ ]:
# 3. Cross-layer attention coherence
coh = cross_layer_attention_coherence(model, tokens)
print(f"Mean coherence: {coh['mean_coherence']:.4f}")
print(f"Most coherent transition: L{coh['most_coherent_transition']}->{coh['most_coherent_transition']+1}")
print(f"Least coherent transition: L{coh['least_coherent_transition']}->{coh['least_coherent_transition']+1}")
for t in range(len(coh['layer_coherence'])):
    print(f"  L{t}->{t+1}: {coh['layer_coherence'][t]:.4f}")

In [ ]:
# 4. Source attribution
src = attention_source_attribution(model, tokens, target_pos=-1)
print(f"Top sources per layer: {src['top_sources'].tolist()}")
print(f"\nCumulative source attribution:")
for s in range(len(tokens)):
    bar = '#' * int(src['cumulative_attribution'][s] * 50)
    print(f"  pos {s}: {src['cumulative_attribution'][s]:.4f} {bar}")

In [ ]:
# 5. Information routing summary
route = attention_information_routing(model, tokens)
for l in range(cfg.n_layers):
    print(f"Layer {l}: mean_dist={route['mean_attention_distance'][l]:.2f}, "
          f"concentration={route['attention_concentration'][l]:.3f}, "
          f"self={route['self_attention_fraction'][l]:.3f}, "
          f"local={route['forward_attention_fraction'][l]:.3f}")